# ATC / ADAPT — Colab: SFT (JSON) + GRPO for medium tier and negotiation

This notebook follows the same layout as `training/atc_multiagent_colab.ipynb` in this repo, and adds an **optional SFT cold-start** phase (`training/build_sft_dataset.py` + `training/train_sft.py`) before **GRPO** (`training/train_grpo.py`).

**Reference notebook (your link):** [Google Colab drive](https://colab.research.google.com/drive/11C0POK25U1hTRi-KwIV3HaYYYCWJL7tL?usp=sharing)

**Use case:** improve Qwen 2.5–7B on Tier 1–3 and negotiation (e.g. `mumbai_bank_balance_medium`, ADAPT ICU) where zero-shot JSON and coordination break down.

**Runtime:** `Runtime` → `Change runtime type` → **GPU** (T4 works for short runs; L4/A100 recommended for 7B + 200+ GRPO episodes).


## 0 — Colab secrets (recommended)

Colab sidebar → **Secrets** → create `HF_TOKEN` for Hugging Face downloads and HF Inference Router (if you use API inference later).


In [ ]:
try:
    from google.colab import userdata
    import os
    t = userdata.get("HF_TOKEN")
    if t:
        os.environ["HF_TOKEN"] = t
        os.environ["HUGGING_FACE_HUB_TOKEN"] = t
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("No HF_TOKEN secret — add one for gated models / router.")
except Exception as e:
    print("Secrets unavailable (not on Colab?)", e)


## 1 — Paths and hyperparameters

Edit `REPO_URL` to your GitHub fork if the default repo is wrong. With `USE_DRIVE=True`, checkpoints land under **Google Drive** so disconnects do not wipe training.


In [ ]:
from pathlib import Path
import os

USE_DRIVE = True
# Default: public repo from project README (change to your fork URL if needed)
REPO_URL = "https://github.com/GTsingh12/ATS-openenv.git"
REPO_DIR = Path("/content/ATS")

OUTPUT_DIR = Path("/content/drive/MyDrive/atc-grpo-medium") if USE_DRIVE else Path("/content/atc-grpo-medium")
SFT_DATA = Path("/content/drive/MyDrive/atc-sft-data.jsonl") if USE_DRIVE else Path("/content/atc-sft-data.jsonl")
SFT_ADAPTER_DIR = Path("/content/drive/MyDrive/atc-sft-json-adapter") if USE_DRIVE else Path("/content/atc-sft-json-adapter")

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
SEED = 42
LORA_RANK = 16

RUN_SFT = True
SFT_EPISODES = 800
SFT_EPOCHS = 1.0
GRPO_EPISODES = 150
EVAL_EPISODES = 8

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("OUTPUT_DIR", OUTPUT_DIR)
print("RUN_SFT", RUN_SFT, "GRPO_EPISODES", GRPO_EPISODES)


In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SFT_ADAPTER_DIR.parent.mkdir(parents=True, exist_ok=True)
print("Ready:", OUTPUT_DIR)


## 2 — Clone repository


In [ ]:
import shutil, subprocess, os, sys
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("cwd:", Path.cwd())


## 3 — Install dependencies (Unsloth + TRL)

If you hit import/version errors after this cell, use **Runtime → Restart session** and re-run from §2.


In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *pkgs], check=True)

pip("pip")
pip("unsloth[colab-new]@git+https://github.com/unslothai/unsloth.git")
pip("trl>=0.9.6", "datasets>=2.20.0", "accelerate>=0.32.0", "peft>=0.12.0", "bitsandbytes>=0.43.0")
pip("matplotlib>=3.9.0", "numpy>=1.26.0")
pip("openenv-core[core]>=0.2.3", "fastapi>=0.128.0", "openai>=2.30.0", "pydantic>=2.12.0", "uvicorn>=0.41.0")
print("pip install complete")


## 4 — GPU check + GRPO dataset sanity


In [ ]:
import torch
from training.dataset import build_episode_dataset

assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type → GPU)."
print("GPU:", torch.cuda.get_device_name(0))

rows = build_episode_dataset(
    n_episodes=4,
    seed=SEED,
    include_adapt=True,
    include_generator=False,
    include_supervisor=False,
)
print("rows:", len(rows), "roles:", sorted({r["agent_role"] for r in rows}))


## 5 — Phase A (optional): SFT on strict JSON

Builds `SFT_DATA` jsonl with **teacher** heuristics, then runs `training/train_sft.py` into `SFT_ADAPTER_DIR`. This targets **parseable** AMAN/DMAN/ADAPT JSON before RL.


In [ ]:
import subprocess, sys

if not RUN_SFT:
    print("Skipping SFT (set RUN_SFT=True to enable).")
else:
    subprocess.run([
        sys.executable, "training/build_sft_dataset.py",
        "--out", str(SFT_DATA),
        "--episodes", str(SFT_EPISODES),
        "--seed", str(SEED),
    ], cwd=str(REPO_DIR), check=True)
    subprocess.run([
        sys.executable, "training/train_sft.py",
        "--dataset", str(SFT_DATA),
        "--output_dir", str(SFT_ADAPTER_DIR),
        "--model", MODEL_NAME,
        "--epochs", str(SFT_EPOCHS),
        "--batch_size", "1",
        "--grad_accum", "8",
        "--lora_rank", str(LORA_RANK),
        "--seed", str(SEED),
    ], cwd=str(REPO_DIR), check=True)
    print("SFT adapter:", SFT_ADAPTER_DIR)


## 6 — Phase B: GRPO multi-agent training

Runs the full `training/train_grpo.py` loop (curriculum generator, AMAN/DMAN rewards, optional ADAPT in dataset). **Increase `GRPO_EPISODES`** on larger GPUs for better medium-tier convergence.

Note: GRPO currently starts a **fresh** LoRA on the base model. The SFT adapter in §5 is still valuable for **separate** inference or future `train_grpo` init-adapter support; see README “SFT cold-start”.


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "training/train_grpo.py",
    "--model", MODEL_NAME,
    "--output_dir", str(OUTPUT_DIR),
    "--episodes", str(GRPO_EPISODES),
    "--lora_rank", str(LORA_RANK),
    "--seed", str(SEED),
    "--eval_episodes", str(EVAL_EPISODES),
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)
print("Saved:", OUTPUT_DIR)


## 7 — Evaluation: base vs trained (`training/eval.py`)


In [ ]:
import subprocess, sys
from pathlib import Path

eval_json = OUTPUT_DIR / "eval_results_colab.json"
subprocess.run([
    sys.executable, "training/eval.py",
    "--base", MODEL_NAME,
    "--trained", str(OUTPUT_DIR),
    "--episodes", str(EVAL_EPISODES),
    "--output", str(eval_json),
], cwd=str(REPO_DIR), check=True)
print(eval_json.read_text()[:2000])


## 8 — Reward curves and eval plots


In [ ]:
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

for argv in (
    ["training/plot_rewards.py", "--input", str(OUTPUT_DIR / "reward_curves.json"), "--save", str(plots_dir), "--no_show"],
    ["training/plot_rewards.py", "--eval_results", str(OUTPUT_DIR / "eval_results.json"), "--save", str(plots_dir), "--no_show"],
):
    subprocess.run([sys.executable] + argv, cwd=str(REPO_DIR), check=False)

for name in ("training_curves.png", "eval_comparison.png"):
    p = plots_dir / name
    if p.exists():
        display(Image(filename=str(p)))


## 9 — Optional: push adapter to Hugging Face Hub

Uncomment and set `HUB_REPO_ID` after `huggingface_hub` login.


In [ ]:
# from huggingface_hub import HfApi
# import os
# HUB_REPO_ID = "your-user/atc-grpo-medium-7b"
# api = HfApi(token=os.environ.get("HF_TOKEN"))
# api.upload_folder(folder_path=str(OUTPUT_DIR), repo_id=HUB_REPO_ID, repo_type="model")
# print("Uploaded to", HUB_REPO_ID)
print("Hub upload cell is commented out by default.")


## Troubleshooting

| Issue | What to do |
|-------|------------|
| CUDA OOM on 7B | Lower `GRPO_EPISODES`; in SFT use `--batch_size 1`; try smaller model or A100 |
| `trl` / `GRPOConfig` keyword errors | Restart runtime; pin `trl` to version compatible with your Unsloth build |
| Clone permission denied | Use a **public** `REPO_URL` or embed a read token in the git URL |
| `eval.py` cannot load trained folder | Ensure `OUTPUT_DIR` contains `adapter_config.json` from GRPO save |

**Local MetaHack repo:** If your canonical code lives at `F:\College\SEM4\MetaHack\ats`, push it to GitHub and set `REPO_URL` to that remote so Colab matches your branch.
